## Extract Refunds Data via JDBC

This notebook extracts refunds data from Azure SQL Database using JDBC and loads it into the bronze layer (`gizmobox.bronze.refunds`) as a Delta table.

In [0]:
# Set up the Unity Catalog namespace
spark.sql("USE CATALOG gizmobox")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
print("Catalog and schema configured")

#### What this JDBC code does

This code reads the `Refunds` table from an Azure SQL Database into a PySpark DataFrame called `refunds_df`.

* `spark.read` starts the data-reading process.
* `format("jdbc")` tells Spark to use a JDBC database connection.
* `url` gives the database server and database name.
* `dbtable` tells Spark which table to read.
* `user` and `password` provide the login details.
* `load()` runs the connection and brings the data into Spark.

In short, this block connects to Azure SQL, reads the refunds data, and stores it in `refunds_df` so it can be transformed or saved in Databricks.

For production use, credentials should be stored in Databricks secrets instead of being written directly in the notebook.

In [0]:
# Connect to Azure SQL Database and read refunds table
refunds_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:sqlserver://gizmobox-db1.database.windows.net:1433;database=gizmoboxdb;") \
    .option("dbtable", "dbo.Refunds") \
    .option("user", "gizmoboxadm") \
    .option("password", "Arthi@123456") \
    .load()


# Write to bronze layer as Delta table
refunds_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gizmobox.bronze.py_refunds")

print("✓ Refunds data successfully loaded to gizmobox.bronze.py_refunds")

In [0]:
# Preview the loaded refunds data
refunds_preview = spark.table("gizmobox.bronze.py_refunds")
display(refunds_preview)